# ConvexPi — Julia starter

Write a quant strategy in **Julia** and submit it. The grader runs your Julia `on_day` over a hidden
out-of-sample period and scores it with the *same* engine as Python/R.


In [ ]:
# 1. Get the exact market the grader uses (deterministic from the seed — exported via Python).
run(`pip install -q convexpi-lab`)
run(`python -c "from convexpi.lab import SyntheticMarket; from convexpi.lab.multilang import export_market; export_market(SyntheticMarket(seed=42), \"market\")"`)


In [ ]:
# 2. Load the in-sample data to fit on.
using DelimitedFiles
readmat(p) = readdlm(p, ',', Float64)
prices = readmat("market/train/prices.csv")
feat_names = strip.(readlines("market/train/feature_names.txt"))
features = Dict(n => readmat(joinpath("market/train/features", n * ".csv")) for n in feat_names)
println("prices: ", size(prices), " | features: ", join(feat_names, ", "))


## 3. Write your strategy

Define `on_day(day, features, prices, portfolio)` returning target weights (one per stock) — exactly
what the grader runs. Keep it as a string so you can both test it and submit it.


In [ ]:
strategy_code = """
function on_day(day, features, prices, portfolio)
    sig = copy(features["mom_1m"])     # 1-month momentum, z-scored across stocks
    sig[.!isfinite.(sig)] .= 0.0
    s = sum(abs.(sig))
    return s > 0 ? sig ./ s : sig      # long winners / short losers, gross leverage 1
end
"""
include_string(Main, strategy_code)    # define on_day locally


In [ ]:
# 4. (optional) sanity-check on one in-sample day
d = 300
w = on_day(d, Dict(n => vec(m[d, :]) for (n, m) in features), vec(prices[d, :]), zeros(size(prices, 2)))
println("weights sum to ", sum(w), " over ", length(w), " stocks")


## 5. Submit

Create an API key at **convexpi.ai/settings/api-keys** and set it below. Your OOS Sharpe appears on
the leaderboard within a few minutes.


In [ ]:
using Pkg; Pkg.add(["HTTP", "JSON"]); using HTTP, JSON
function submit_strategy(name, code; slug="demo-fall-2026")
    key = ENV["CONVEXPI_API_KEY"]
    body = JSON.json(Dict("slug" => slug, "strategyName" => name, "code" => code, "language" => "julia"))
    r = HTTP.post("https://www.convexpi.ai/api/submissions",
                  ["Authorization" => "Bearer " * key, "Content-Type" => "application/json"],
                  body; status_exception=false)
    println(r.status, " ", String(r.body))
end
ENV["CONVEXPI_API_KEY"] = "cpk_..."   # <- your key
submit_strategy("my-julia-momentum", strategy_code)
